In [1]:
import pandas as pd
import numpy as np

#### Training a Neural Network
1. Create a model
2. Choose a loss function
3. Define a dataset
4. Set an optimizer
5. Run a training loop

In [2]:
animals = pd.read_csv("/Users/srisuphachawla/Downloads/zoo.csv.xls")

In [3]:
animals.head()

,animal_name,hair,feathers,eggs,milk,airborne,aquatic,predator,toothed,backbone,breathes,venomous,fins,legs,tail,domestic,catsize,class_type
0,aardvark,1,0,0,1,0,0,1,1,1,1,0,0,4,0,0,1,1
1,antelope,1,0,0,1,0,0,0,1,1,1,0,0,4,1,0,1,1
2,bass,0,0,1,0,0,1,1,1,1,0,0,1,0,1,0,0,4
3,bear,1,0,0,1,0,0,1,1,1,1,0,0,4,0,0,1,1
4,boar,1,0,0,1,0,0,1,1,1,1,0,0,4,1,0,1,1


In [4]:
# Animal name not needed - names do not determine classification
features = animals.iloc[:, 1: -1]

# converting into numpy array for easier handling with pytorch
X = features.to_numpy()
print(X)

[[1 0 0 ... 0 0 1]
 [1 0 0 ... 1 0 1]
 [0 0 1 ... 1 0 0]
 ...
 [1 0 0 ... 1 0 1]
 [0 0 1 ... 0 0 0]
 [0 1 1 ... 1 0 0]]


In [5]:
# define target values (ground truth)

target = animals.iloc[: , -1]
y = target.to_numpy()
print(y)

[1 1 4 1 1 1 1 4 4 1 1 2 4 7 7 7 2 1 4 1 2 2 1 2 6 5 5 1 1 1 6 1 1 2 4 1 1
 2 4 6 6 2 6 2 1 1 7 1 1 1 1 6 5 7 1 1 2 2 2 2 4 4 3 1 1 1 1 1 1 1 1 2 7 4
 1 1 3 7 2 2 3 7 4 2 1 7 4 2 6 5 3 3 4 1 1 2 1 6 1 7 2]


In [6]:
import torch
# allows us to store X and y as tensors, easier to manage
from torch.utils.data import TensorDataset

# Instantiate datatset class
dataset = TensorDataset(torch.tensor(X), torch.tensor(y))

# Access an individual sample
input_sample, label_sample = dataset[0]
print("input sample: ", input_sample)
print("label sample: ", label_sample)

input sample:  tensor([1, 0, 0, 1, 0, 0, 1, 1, 1, 1, 0, 0, 4, 0, 0, 1])
label sample:  tensor(1)


In [7]:
from torch.utils.data import DataLoader

# how many samples in each iter, DL models -> large models -> batching helps process large datasets at once
# shuffle randomizes the data order at each epoch, improving model generalization 
# Epoch = one full pass thru the training dataloader
# Generalization : model performs well on unseen data, rather than just memorizing the training set


# create a dataloader
dataloader = DataLoader(dataset, batch_size= 2, shuffle=True)

for batch_inputs, batch_labels in dataloader:
    print('batch_inputs', batch_labels)
    print('batch_labels: ', batch_labels)

batch_inputs tensor([5, 1])
batch_labels:  tensor([5, 1])
batch_inputs tensor([6, 1])
batch_labels:  tensor([6, 1])
batch_inputs tensor([2, 1])
batch_labels:  tensor([2, 1])
batch_inputs tensor([4, 1])
batch_labels:  tensor([4, 1])
batch_inputs tensor([5, 6])
batch_labels:  tensor([5, 6])
batch_inputs tensor([6, 1])
batch_labels:  tensor([6, 1])
batch_inputs tensor([1, 2])
batch_labels:  tensor([1, 2])
batch_inputs tensor([7, 4])
batch_labels:  tensor([7, 4])
batch_inputs tensor([1, 1])
batch_labels:  tensor([1, 1])
batch_inputs tensor([7, 2])
batch_labels:  tensor([7, 2])
batch_inputs tensor([1, 1])
batch_labels:  tensor([1, 1])
batch_inputs tensor([1, 1])
batch_labels:  tensor([1, 1])
batch_inputs tensor([1, 4])
batch_labels:  tensor([1, 4])
batch_inputs tensor([1, 3])
batch_labels:  tensor([1, 3])
batch_inputs tensor([6, 7])
batch_labels:  tensor([6, 7])
batch_inputs tensor([7, 6])
batch_labels:  tensor([7, 6])
batch_inputs tensor([1, 5])
batch_labels:  tensor([1, 5])
batch_inputs t

### Run a Training Loop
- calculate loss (forward pass)
- compute gradients (backpropagation)
- update model params

 

In [8]:
import torch.nn as nn
import torch.optim as optim

def mean_squared_loss(prediction, target):
    return np.mean((prediction - target) ** 2)

criterion = nn.MSELoss()
loss = criterion(prediction, target)

NameError: name 'prediction' is not defined

In [9]:
# create the dataset and dataloader

# convert pandas into numpy first 
X = torch.tensor(features.to_numpy()).float()
y = torch.tensor(target.to_numpy()).float().unsqueeze(1)

# organize into right data types 
dataset = TensorDataset(X,y)

# load into dataloader to enable batching, batch_size is customizable
dataloader = DataLoader(dataset, batch_size = 4, shuffle = True)

# create the model 
model = nn.Sequential(nn.Linear(16, 2),
                      nn.Linear(2,1))

# Create the loss and optimizer, default lr of 0.001 for DL problems
criterion = nn.MSELoss()
optimizer = optim.SGD(model.parameters(), lr = 0.001)


In [10]:
num_epochs = 100
 
for epoch in range(num_epochs):
    for data in dataloader:
        # set the gradients to 0
        optimizer.zero_grad()

        # get features and target from the data loader
        features, target = data

        # run a forward pass
        pred = model(features)

        loss = criterion(pred, target)
        loss.backward()

        # update params
        optimizer.step()


#### Limitations of Sigmoid Function
- outputs bounded between 0 and 1
- usable anywhere in a network but the gradients are very small for large and small valuses of x
- cause saturation, leading to the vanishing gradients problem (each gradient depends on the previous one, small gradient -> fail to update the next one properly)

--> Softmax also suffers through the same, 

##### Hence, these two activation functions are not ideal for hidden layers and best for last layer only

#### Acitivation for Hidden or linear layer: RELU
- Rectifies Linear Unit 
- for positive inputs, output equals input
- for negative inputs, output is 0 
- helps overcome vanishing gradients
- nn.RelU()


#### Leakuy ReLu
- Positive - behaves like Relu
- Negative - scaled by a small coefficient (default = 0.01)
- graidnets for negative inputs are non-zero

- nn.LeakyReLu(negative_slope = 0.05)

In [11]:
# eg: 

# Create a ReLU function with PyTorch
relu_pytorch = nn.ReLU()

x_pos = torch.tensor(2.0)
x_neg = torch.tensor(-3.0)

# Apply the ReLU function to the tensors
output_pos = relu_pytorch(x_pos)
output_neg = relu_pytorch(x_neg)

print("ReLU applied to positive value:", output_pos)
print("ReLU applied to negative value:", output_neg)


ReLU applied to positive value: tensor(2.)
ReLU applied to negative value: tensor(0.)


#### Learning Rate and Momentum
- training a neural network - solving an optimization problem ( SGD optimizer)

sgd = optim.SGD(model.parameters(), lr = 0.01, momentum = 0.95)

Two arguments
- learning rate: controls the step size
- momentum: adds inertia to avoid getting stuck


- step size decrease near zero as the gradients get smaller with optimal learning rate
- small learning rate, optimzer takes longer to find the minimum
- big learnin rate - optimer bounces back and fortch and loses sight of the minimum

#### Without Momentum 
- lr = 0.01, momentum 0 (optimizer gets stuck at the first dip of the function)

- momentum = 0.9, --> good 

#### Layer Initialization
- a layer weihts are initialized to small values

In [ ]:
layer = nn.Linear(64, 128)
print(layer.weight.min(), layer.weight.max())
# the output is a weightum sum of inputts from the previous layer
# keepin g both the inpur data and layer weights small ensures stable outputs


tensor(-0.1249, grad_fn=<MinBackward1>) tensor(0.1250, grad_fn=<MaxBackward1>)


In [ ]:
# use uniform initialization for layer0 and layer1 weights
layer = nn.Linear(64, 128)
nn.init.uniform_(layer.weight)

print(layer.weight.min(), layer.weight.max())
# weights value now range from 0 to 1

tensor(8.0109e-05, grad_fn=<MinBackward1>) tensor(0.9999, grad_fn=<MaxBackward1>)


#### Transfer Learnin
- takes a model that was trained on a first task and reuses for a second similar task
eg: traininf a model on US salaries dataset, now we have new dataset based on UK dataset, reusue weights to train on new on 

In [ ]:
layer == nn.Linear(64, 128)
# torch.save(layer, 'layer.pth')

# new_layer = torch.load('layer.pth')

#### Fine Tuning
- a type of transfer learning
- load weights from previous model but with a smaller learning rate
- train part of the network (freeze some of them)
- freeze early layers of network and fine tune layers closer to output layer


In [ ]:
model = nn.Sequential(nn.Linear(64, 128), nn.Linear(128, 256))

for name, param in model.named_parameters():
    # check for first layer's weight
    if name == '0.weight' :
        # Freeze the weight
        param.requires_grad = False

    # Check for second layer's weight
    if name == '1.weight':

        # Freeze this weight
        param.requires_grad = False



### Evaluation of Models
- training : adjust model params
- validation : tunes hyperparams
- test : evaluate final model performance


----> calculate training Loss

for each epoch, sum the loss across all batches in the dataloader
- compute the mean training loss at the end of the epoch

In [ ]:
training_loss  = 0.0

for inputs, labels in trainloader:
    # run the forward pass
    outputs = model(inputs)

    # compute the loss
    loss = criterion(outputs, labels)

    # backpropagation 
    loss.backward()   # compute gradients
    optimizer.step()  # update weights
    optimizer.zero_grad()  # reset gradients

    # calculate and sum the loss
    training_loss += loss.item()
epoch_loss = training_loss / len(trainloader)

In [ ]:
# calculating validation loss

validation_loss = 0.0
model.eval() # put model in evaluation mode

with torch.no_grad(): # disable gradients for efficiency
    for inputs, labels in validationloader:
        # run the forward pass
        outputs = model(inputs)

        # calculate the loss
        loss = criterion(outputs, labels)
        validation_loss += loss.item()

epoch_loss = validation_loss / len(validationloader) # compute mean loss
model.train() # switch back to training mode

# overfit model.- training loss decreases, validation loss rises 
# ie the model is learning the trainign data too well and wont perform weel on new data
# doesnt always reflect how accurately it makes predcition

In [ ]:
### calc accuracy woth torchmetrics

import torchmetrics

metric = torchmetrics.Accuracy(task = "multiclass", num_classes = 3)

for features, labels in dataloader:
    outputs: model(features) # forwardpass

    # compute batch accuracy (keeping argmax for one hot labels)
    metric.update(outputs, labels.argmax(dim=-1))
    
# compute accuracy over the whole epoch
accuracy = metric.compute()

# reset for next epoch
metric.reset()

#### fighting overfitting 
- reasons for overfitting 
   - dataset is not large enough ( get more data )
   - model has too much capacity ( reduce model size, add dropout)
   - weights are too large (weight decay to force params to remain small)
   

In [ ]:
# regularization using a dropout layer 
# randomly zeroes out elements of the input tensor during training

model = nn.Sequential(nn.Linear(8, 4),
                      nn.ReLU(),
                      nn.Dropout(p=0.5))

features = torch.randn((1, 8))
print(model(features))
# dropout is added after the activation function
# model.train() for training
# model.eval() to disable dropout during evaluation 


tensor([[0.0000, 0.9601, 0.7943, 0.0000]], grad_fn=<MulBackward0>)


In [ ]:
# example: 

model = nn.Sequential(
    nn.Linear(8, 6),
    nn.Linear(6, 4),
    nn.Dropout(p=0.5))

model.train()
output_train = model(features)

# Forward pass in evaluation mode (Dropout disabled)
model.eval()
output_eval = model(features)

# Print results
print("Output in train mode:", output_train)
print("Output in eval mode:", output_eval)

In [ ]:
# Regularization with weight decay
optimizer = optim.SGD(model.parameters(), lr = 0.001, weight_decay = 0.0001)

# controlled by the weight decay param in the optimizer, typically set to a small val  - 0.0001
# weight decay encourages smaller weights by adding a penakuty during optimization
# helps reduce overfitting, keeping weights smalller and imporiving generalization
# higher weight decay -> stronger the regularization (less likely to overfit)

#### Data Augmentation 
- new data is expensive
- we have a way to expand data artificially
- diff views through rotation and scaled

In [ ]:
# modify the training loop to overfit a single data point
 
features, labels = next(iter(dataloader))

for i in range(1000):
    outputs = model(features)
    loss = criterion(outputs, labels)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

# should reach 1.0 accuracy and 0 loss if nodel is set up correctly

# step 2: reduce overfitting - maximize validation accuracy
# keep track of each hyperparams and validation accuracy


# step 3: fine tune hyperparameters
# grid search

for factor in range(2, 6):
    lr = 10 ** -factor

# random search - typically more efficient as it avoids unnecesary tests and increases the chance of finding optimal setting
factor = np.random.uniform(2, 6)
lr = 10 ** -factor 





### summary

Chapter 1:
- created small neural networks  
- linear layers

Chapter 2:
- used loss and activation fucntions
- calc derivatives
- backpropagation

Chapter 3:
- trained a neural network
- learning rate and momentum
- impact of the above

Chapter 4:
- strategies to improve yoru model
- reduced overfitting
- evaluated model performance




## Handling images with Pytorch
cloud tyoe classification/data from kaggle, 7 cloud images 

In [ ]:
from torchvision.datasets import ImageFolder
from torchvision import transforms

# Data augmentation - adding randomg transformations, more robust, reduce overfitting
train_transforms = transforms.Compose([
    # augmentation 1
    transforms.RandomHorizontalFlip(),
    # augmentation 2
    transforms.RandomRotation(45),
    transforms.ToTensor(),
    transforms.Resize((128, 128)),
])

dataset_train = ImageFolder(
    "data/clouds_train",
    transform=train_transforms,
)

# display Images

dataloader_train = DataLoader(
    dataset_train,
    shuffle=True,
    batch_size=1,
)

image, label = next(iter(dataloader_train))
print(image.shape) # output: torch.size([1, 3, 128, 128])

# image = image.squeeze().permute(1, 2, 0)
print(image.shape) # torch.size([128, 128, 3])

import matplotlib.pyplot as plt
plt.imshow(image)
plt.show() # shows the image

FileNotFoundError: [Errno 2] No such file or directory: 'data/clouds_train'

#### why not use linear layers
- slow training 
- overfitting 
- dont recognize spatial patterns
- use convolutional layer!!


#### Convolutional Layer
- sldie filter of params over the input
- at each position, perform convolution
- resulting feature map, preservers spatial patterns from input
- uses fewer params than linear layer
- one filter = one feature map
- apply activations to feature maps
- all feature maps combined form the output


               nn.Conv2d(3, 32, kernel_size= 3)         


#### zero padding
we typically add a frame of zeroes to convolutional layer's input 

nn.Conv2d(
    3, 32, kernel_size = 3, padding=1
)

maintains spatial dimensions of the input and output tensors
ensures border pizels are trated equally to others


#### max pooling 
- slide non-overlapping windown over input
- at each position, retain only the maximum value
- used after convolutional layers to reduce spatial dimensions

In [18]:
class Net(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.feature.extractor = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ELU(),
            nn.MaxPool2d(kernel_size=2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ELU(),
            nn.MaxPool2d(kernel_size=2),
            nn.Flatten(),
        )
        self.classifier = nn.Linear(64*16*16, num_classes)

    def forward(self, x):
        x = self.feature_extractor(x)
        x = self.classifier(x)
        return x

#### Augmentation can be confusing depends on the task --> W (flips to M)

WHats good for cloud classfication?
- random rotation - expose model to different angles of cloud formations
- horizontal flip - simulate different viewpoints of the sky
- auto contrast adjustment - different lighting conditions



#### Cross - Entropy Loss
- binary cross entropy for binary classification
- multi class classification - cross entropy loss
- criterion = nn.CrossEntropyLoss()

In [ ]:
# image classifier training loop 
net = Net(num_classes = 7)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(net.parameters(), lr = 0.001)

for epoch in range(10):
    for images, labels in dataloader_train:
        optimizer.zero_grad()
        outputs = net(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

#### Evaluating image classifiers

---> data Augmentation at test time
- we defined train_tranforms compose in detail for training data but we dont do that for test data

In [ ]:
# for test data

test_transforms = transforms.Compse([
    # mo data augmentation at test time
    transforms.ToTensor(),
    transforms.Resize((64, 64))
])

dataset_test = ImageFolder(
    "clouds_test",
    transform=test_transforms
)

In [19]:
# Binary Classification for Precision and Recall
# Precision - Fraction of correct positive predictions
# Recall - fraction of all positive examples correctly predicted

'''
in multiclass classification, seperate precision and recall for each class

With 7 classes, we have 7 precision and 7 recall scores
- 

use micro when imbalanced datasets
use macro when you care about performance on smaller classes, even with fewer data points
use weighted when you consider errors in larger classes as more important

'''

from torchmetrics import Recall

recall_per_class = Recall(task = "multiclass", num_classes = 7, average = None)
recall_micro = Recall(task = "multiclass", num_classes = 7, average = "micro")
recall_macro = Recall(task = "multiclass", num_classes = 7, average = "macro")
recall_weighted = Recall(task = "multiclass", num_classes = 7, average = "weighted")

# evaluation 
from torchmetrics import Precision, Recall

metric_precision = Precision(
    task = "multiclass", num_classes = 7, average = "macro"
)
metric_recall = Recall(
    task = "multiclass", num_classes = 7, average = "macro"
)

net.eval()

with torch.no_grad():
    for images, labels in dataloader_test:
        outputs = net(images)
        _, preds = torch.max(outputs, 1)
        metric_precision(preds, labels)
        metric_recall(preds, labels)

precision = metric_precision.compute()
recall = metric_recall.compute()

print(f"Precision: {precision}")
print(f"Recall: {recall}")

# using larger images, more convolutonal layers, and a classifier with more than one linear layer could improve both Precision and Recall

ModuleNotFoundError: No module named 'torchmetrics'

In [20]:
## Analyzing Performance per class
# computer metric with average = None
metric_recall = Recall(
    task = "multiclass", num_classes = 7, average = None
)
# gets one score per class
net.eval()
with torch.no_grad():
    for images, labels in dataloader_test:
        outputs = net(images)
        _, preds = torch.max(outputs, 1)
        metric_recall(preds, labels)

recall = metric_recall.compute()
print(f"Recall: {recall}")
# you get 7 answers for 7 images


dataset_test.class_to_idx # attribute maps class names to indices

# dict comprehension 
# tensor of length 1, turned into scalar 
# 1.0 - classified correctly 
# lowest recall score - hardest to classify
{
    k: recall[v].item()
    for k, v
    in dataset_test.class_to_idx.items()
}


NameError: name 'Recall' is not defined

### Handling Sequences with Pytorch
- with sequential data, spliiting data randomly creates. "Look ahead Bias"
- better: split by time (training data: first 3 years, test: 4th year)

In [ ]:
def create_sequences(df, seq_length):
    xs, ys = [], []

    for i in range(len(df) - seq_length):
        x = df.iloc[i: (i + seq_length), 1]
        y = df.iloc[i + seq_length, 1]
        xs.append(x)
        ys.append(y)


X_train, y_train = create_sequences(train_data, seq_length = 24*4)
print(X_train.shape, y_train.shape)

# convert to torch dataset

from torch.utils.data import TensorDataset

dataset_train = TensorDataset(
    torch.from_ numpy(X_train).float(),
    torch.from_numpy(y_train).float(),
)

# above can be applied to LLMs, speech recognition as well

#### Recurrent Neural Networks
- RNNs have connections poinitn gback 
- feed forwards networks
- nn.RNN()
- input x, output y, hidden state h

there is also Deep RNNS
- pass through multiples neurons one after another

### Sequence to Sequence Architecture
- pass sequence as input, use the entire output sequence
- eg: real time speech recognition 

### Sequence to Vector Architecture
- use only the last output
- eg: Text topic classification

### Vector to Sequence Architecture
- pass single intput, use the entire output sequence
- text generation

### Encoder Decoder Architecture
- pass entire input sequence, only then start using output sequence
- eg: Machine translation

In [ ]:
# Sequence to Vector RNN

class Net(nn.Module):
    def __init__(self):
        super().__init__():
        self.rnn = nn.RNN(
            input_size =1,
            hidden_size=32,
            num_layers=2,
            batch_first=True
        )
        self.fc = nn.Linear(32, 1) # linear layer

    def forward(self, x): # init first hidden state to zeros
        # pass input and first hidden state thru RNN layer
        # select last RNN's output and pass it through linear layer
        h0 = torch.zeros(2, x.size(0), 32)
        out, _ = self.rnn(x, h0)
        out = self.fc(out[:, -1, :])
        return out


### LSTM and GRU cells
- RNN cells maintain memory via hidden state
- this memory is very short term 
- solution: LSTM and GRU cells

----> RNN Cells
- two inputs (current input data (x) and previous hidden state)


----> LSTM cell 
- Three inputs and outputs (two hidden states) 
   h: short term state
   c: long term state

- Three gates
   Forget gate: what to remove from long term memory
   Input gate: what to save to long term memory
   output gate; what to return at the current time step 



------>. GRU cell
- simplified version of LSTM
- just one hidden state


#### which to use?
RNN - not used much tehse days, short term memory problem

LSTM -

GRU - less complex and less computation, 

relative performance varies per use case between GRU and LSTM. Good practice to try out both

In [ ]:
# LSTM 
class Net(nn.Module):
    def __init__(self):
        super().__init__():
        self.lstm = nn.LSTM(
            input_size =1,
            hidden_size=32,
            num_layers=2,
            batch_first=True
        )
        self.fc = nn.Linear(32, 1) # linear layer

    def forward(self, x): 
        h0 = torch.zeros(2, x.size(0), 32)
        c0 = torch.zeros(2, x.size(0), 32)
        out, _ = self.lstm(x, (h0, c0))
        out = self.fc(out[:, -1, :])
        return out

In [ ]:
# GRU 
class Net(nn.Module):
    def __init__(self):
        super().__init__():
        self.gru = nn.GRU(
            input_size =1,
            hidden_size=32,
            num_layers=2,
            batch_first=True
        )
        self.fc = nn.Linear(32, 1) # linear layer

    def forward(self, x): 
        h0 = torch.zeros(2, x.size(0), 32)
        c0 = torch.zeros(2, x.size(0), 32)
        out, _ = self.gru(x, h0)
        out = self.fc(out[:, -1, :])
        return out

### Training and evaluating RNN

--> expanding tensors
recurrent layers expect input shape (batch_size, seq_length, num_features)



In [ ]:
for seqs, labels in dataloader_train:
     print(seqs.shape)


seqs = seqs.view(32, 96, 1)
print(seqs.shape)

#### squeezing tensors
in evaluation loop, we need to revirt the reshaping done in the training loop

labels are of shpe (batch_size)

for seqs, labels in test_loader:
     print(labels.shape)

model outputs are (batchsize, 1)
out = net(seqs)

--- shapes od model outputs and labels must match for the loss function 
- we can drop the last dimension from model outputs
    out = net(seqs).squeeze()


In [ ]:
## training loop

net = Net()

criterion = nn.MSELoss()
optimizer - optim.Adam(net.parameters(), lr = 0.001)

for epoch in range(num_epochs):
    for seqs, labels in dataloader_train:
        seqs = seqs.view(32, 96, 1)
        outputs = net(seqs)
        loss = criterion(outputs, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()


# evaluation loop

mse = torchmetrics.MeanSquaredError() # set up mse metric

net.eval()
with torch.no_grad():
    # iterate thru test data with no gradients
    # reshape model inputs
    # squeeze model outputs
    # update the metric
    # computer final metric value
    for seqs, labels in test_loader: 
        seqs = seqs.view(32, 96, 1)
        outputs = net(seqs).squeeze()
        mse(outputs, labels)

print(f"Test MSE: {mse.compute()}")



### Multi Input Models
- using multiple sources of information 
- multi-model models
- metric learning
- supervised learning

In [ ]:
class OmniglotDataset(Dataset):
    def __init__(self, transform, samples):
		# Assign transform and samples to class attributes
        self.transform = transform
        self.samples = samples
                    
        # print(samples[0])
        
    def __len__(self):
		# Return number of samples
        return len(self.samples)

    def __getitem__(self, idx):
        # Unpack the sample at index idx
        img_path, alphabet, label = self.samples[idx]
        img = Image.open(img_path).convert('L')
        # Transform the image 
        img_transformed = self.transform(img)
        return img_transformed, alphabet, label

In [ ]:
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        # Define sub-networks as sequential models
        self.image_layer = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.MaxPool2d(kernel_size=2),
            nn.ELU(),
            nn.Flatten(),
            nn.Linear(16*32*32, 128)
        )
        self.alphabet_layer = nn.Sequential(
            nn.Linear(30, 8),
            nn.ELU(), 
        )
        self.classifier = nn.Sequential(
            nn.Linear(128 + 8, 964), 
        )
    
    # two architecture loop
    def forward(self, x_image, x_alphabet):
		# Pass the x_image and x_alphabet through appropriate layers
        x_image = self.image_layer(x_image)
        x_alphabet = self.alphabet_layer(x_alphabet)
        # Concatenate x_image and x_alphabet
        x = torch.cat((x_image, x_alphabet), dim=1)
        return self.classifier(x)

In [ ]:
# training. loop

net = Net()
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(net.parameters(), lr = 0.01)

for epoch in range(10):
    for img, alpha, labels in dataloader_train:
        optimizer.zero_grad()
        outputs = net(img, alpha)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

TypeError: Net.__init__() missing 1 required positional argument: 'num_classes'

### Multi - output models
- multi task learning (predict two things from the same input)


In [ ]:
# character and alphabet classification

class OmniglotDataset(Dataset):
    def __init__(self, transform, samples):
		# Assign transform and samples to class attributes
        self.transform = transform
        self.samples = samples
                    
        # print(samples[0])
        
    def __len__(self):
		# Return number of samples
        return len(self.samples)

    def __getitem__(self, idx):
        # Unpack the sample at index idx
        img_path, alphabet, label = self.samples[idx]
        img = Image.open(img_path).convert('L')
        # Transform the image 
        img = self.transform(img)
        return img, alphabet, label

In [ ]:
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        # Define sub-networks as sequential models
        self.image_layer = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.MaxPool2d(kernel_size=2),
            nn.ELU(),
            nn.Flatten(),
            nn.Linear(16*32*32, 128)
        )
        self.classifier_alpha = nn.Linear(128, 30)
        self.classifier_char = nn.Linear(128, 964)
    
    # two architecture loop
    def forward(self, x_image, x_alphabet):
        x_image = self.image_layer(x_image)
        output_alpha = self.classifier_alpha(x_image)
        output_char = self.classifier_char(x_image)
        return output_alpha, output_char

In [ ]:
# training. loop

net = Net()
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(net.parameters(), lr = 0.01)

for epoch in range(10):
    for img, labels_alpha, labels_char in dataloader_train:
        optimizer.zero_grad()
        outputs_alpha, outputs_char = net(img)
        loss_alpha = criterion(outputs_alpha, labels_alpha)
        loss_char = criterion(outputs_char, labels_char)
        loss = loss_alpha + loss_char
        loss.backward
        optimizer.step()

TypeError: Net.__init__() missing 1 required positional argument: 'num_classes'

# evaluation of multi output model and loss weighing



In [ ]:
acc_alpha = accuracy(task = "multiclass", num_classes = 30)
acc_char = accuracy(task = "multiclass", num_classes = 964)

net.eval()
with torch.no_grad():
    for images, labels_alpha, labels_char \
    in dataloader.test:
        out_alpha, out_char = net(images)
        _, pred_alpha = torch.max(out_alpha, 1)
        _, pred_char = torch.max(out_char, 1)
        acc_alpha(pred_alpha, labels_alpha)
        acc_char(pred_char, labels_char )

print(acc_alpha.compute())
print(acc_char.compute())


In [ ]:

for epoch in range(10):
    for img, labels_alpha, labels_char in dataloader_train:
        optimizer.zero_grad()
        outputs_alpha, outputs_char = net(img)
        loss_alpha = criterion(outputs_alpha, labels_alpha)
        loss_char = criterion(outputs_char, labels_char)
        loss = loss_alpha + loss_char
        loss.backward
        optimizer.step()

# both classificatontaska deemed equally imp (identiying alohabets and identidying characters)


### varying task importance
- scale more imp loss by a factor of 2
 ( loss = loss_alpha + loss_char *2)

- assign weights that sum to 1
(loss = 0.33 * loss_alpha + 0.67 * loss_char)


but ---> losses on diff scales
- losses must be on the same scale before they are weighted and added
- eg: 
for house prices - > MSE loss
for quality prediction -> cross entropy
- mse loss can reach tens of thousands but cross entropy is typcaully signle digits

- model qould ignore quality assessment task if we combine both 

solution:

In [ ]:
# solution:
loss_price = loss_price / torch.max(loss_price)
loss_quality = loss_quality / torch.max(loss_quality)
loss = 0.7 * loss_price + 0.3 * loss_quality